<a href="https://colab.research.google.com/github/Daniel-534/ModelamientoNCuerpos/blob/main/S_Stars_Dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evolución dinámica e interacciones gravitacionales del cúmulo de estrellas S en el entorno del agujero negro supermasivo Sgr A*

# Modelado N cuerpos, Mecánica Celeste

## Fuentes

### Artículos

* [An Update on Monitoring Stellar Orbits in the Galactic Center](https://arxiv.org/abs/1611.09144)
* [Kinematic Structure of the Galactic Center S-cluster](https://arxiv.org/abs/2006.11399)
* [Sagittarius A* - The Milky Way Supermassive Black Hole](https://arxiv.org/abs/2503.20081)


### Bases de datos

* [25yrs monitoring of stellar orbits in the GC (Gillessen+, 2017)](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/ApJ/837/30)
* [Kinematic structure of the Galactic Center S cluster (Ali+, 2020)](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/ApJ/896/100)

Preguntas de investigación principales:

Dinámica Newtoniana base: ¿Cómo evolucionan las órbitas de las estrellas S (comenzando con un subconjunto de 5 y escalando a 49) al integrar numéricamente sus estados cartesianos iniciales bajo la influencia dominante de Sgr A*?
Perturbaciones de N-cuerpos: ¿Qué tan significativas son las interacciones gravitacionales estrella-estrella en comparación con la atracción central de Sgr A* a lo largo del tiempo? (Comparación entre
N
 simulaciones de 2 cuerpos vs. 1 simulación de
N
+
1
 cuerpos).

In [2]:
!pip install -Uq pymcel celluloid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 9.5 MB/s eta 0:00:00


In [3]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
from celluloid import Camera
import pandas as pd

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!


## Condiciones Iniciales

La base de datos a emplear es un archivo .tsv que contiene información, como los elementos orbitales, de 39 estrellas delcúmulo estelar cercano a Sagitario A* (Sgr A*), el agujero negro supermasivo central de la Vía Láctea, consta de estrellas rápidas llamadas "estrellas S".

In [17]:
# Base de datos de las estrellas del cumulo extraída de nuestro repositorio en GitHub
url = "https://raw.githubusercontent.com/Daniel-534/ModelamientoNCuerpos/main/Sgrestrellas.tsv"

df = pd.read_csv(url, sep=';')

In [18]:
df.head()

,Disk,Name,a,e_a,e,e_e,i,e_i,omega,e_omega,Omega,e_Omega,Tclos,q_Tclos,e_Tclos,Simbad,_RA,_DE
0,black,S1,22.675,0.257,0.665,0.003,121.066,0.401,109.893,0.458,352.484,0.286,2000.261,,0.001,Simbad,266.41684,-29.00787
1,black,S2,5.034,0.001,0.887,0.002,137.514,0.401,73.416,0.745,235.634,1.031,2002.390,,0.020,Simbad,266.41685,-29.00777
2,black,S8,16.637,0.182,0.768,0.022,75.057,0.573,337.931,2.120,317.075,0.630,1979.216,,0.037,Simbad,266.41696,-29.00788
3,black,S9,11.125,0.030,0.791,0.036,81.876,0.458,137.854,0.573,158.079,0.229,1972.924,,0.023,Simbad,266.41690,-29.00790
4,black,S12,11.962,0.105,0.906,0.003,33.060,0.516,311.173,0.802,236.173,1.146,1995.881,,0.001,Simbad,266.41682,-29.00772


Conversión de los elementos orbitales clásicos a estado cartesiano:

In [19]:
# Unidades canónicas para el sistema del centro galáctico
# UL = 1 mpc (miliparsec)  |  UT = 1 año  |  UM = 1 Msol
# G se expresa en mpc³/(Msol·yr²)

G_SI   = 6.674e-11        # m³ kg⁻¹ s⁻²
mpc_m  = 3.0857e13        # metros por miliparsec
Msol   = 1.989e30         # kg por masa solar
yr_s   = 3.1557e7         # segundos por año

G_canon = G_SI * Msol * yr_s**2 / mpc_m**3   # mpc³/(Msol·yr²)
print(f"G = {G_canon:.4e} mpc³/(Msol·yr²)")

# Parámetro gravitacional de Sgr A* (μ = G·M)
M_SgrA  = 4.3e6                   # masas solares
mu_SgrA = G_canon * M_SgrA        # mpc³/yr²
print(f"μ(Sgr A*) = {mu_SgrA:.4f} mpc³/yr²")

# pc.elementos_a_estado(mu, [p, e, i, Ω, ω, f]) → [x, y, z, vx, vy, vz]
# p en mpc → posiciones en mpc, velocidades en mpc/yr

In [20]:
# Seleccionamos las primeras 5 estrellas del cúmulo interno (disco "black")
primeras_5 = df[df['Disk'] == 'black'].head(5).reset_index(drop=True)

estados = []

for _, estrella in primeras_5.iterrows():
    a = estrella['a']          # semi-eje mayor en mpc
    e = estrella['e']          # excentricidad
    p = a * (1 - e**2)        # semilatus rectum en mpc
    i = np.radians(estrella['i'])
    W = np.radians(estrella['Omega'])
    w = np.radians(estrella['omega'])
    f = 0.0                   # condición inicial en el periastro

    elementos = [p, e, i, W, w, f]
    estado = pc.elementos_a_estado(mu_SgrA, elementos)
    estados.append(estado)

In [21]:
columnas_estado = ['x','y','z','vx','vy','vz']

df_estados = pd.DataFrame(estados, columns=columnas_estado)

df_estados

,x,y,z,vx,vy,vz
0,-3.044613,-3.316143,6.118386,-0.425707,0.139084,-0.136456
1,-0.423510,0.092922,0.368220,0.668894,1.657277,0.351110
2,2.364559,-2.709894,-1.401167,0.296348,-0.054754,0.606001
3,1.516981,-0.848134,1.544553,0.580678,-0.134553,-0.644197
4,-1.001365,-0.220053,-0.461718,0.051191,-1.214017,0.467574
5,2.793172,-4.292175,-2.276752,0.406880,0.290751,-0.048959


### Masas
El agujero negro supermasivo Sgr A* en el centro galáctico tiene una masa bien determinada de ~4.15×10^6 M☉ con precisión de ~0.3%
, medida principalmente mediante el seguimiento de órbitas estelares cercanas (Genzel, Ghez, GRAVITY). Por su parte, las estrellas S más luminosas del cúmulo interno son jóvenes de tipo espectral ~B0–B3V. Dado que las masas individuales de las estrellas S se estiman mediante su tipo espectral, y su efecto perturbador es mínimo frente a Sgr A, se asignaron masas estimadas de entre 8 y 15 masas solares basadas en la literatura (Gillessen et al., 2017).

In [29]:
np.random.seed(42)
masas_5 = np.random.uniform(8.0, 14.0, 5)   # Msol, entre 8 y 14 masas solares

print("Masas asignadas a las 5 estrellas (Msol):")
for nombre, masa in zip(primeras_5['Name'].str.strip(), masas_5):
    print(f"  {nombre}: {masa:.2f} Msol")

,Disk,Name,a,e_a,e,e_e,i,e_i,omega,e_omega,Omega,e_Omega,Tclos,q_Tclos,e_Tclos,Simbad,_RA,_DE,Masa_Msol,Masa_kg
0,black,S1,22.675,0.257,0.665,0.003,121.066,0.401,109.893,0.458,352.484,0.286,2000.261,,0.001,Simbad,266.41684,-29.00787,13.227493,2.630948e+31
1,black,S2,5.034,0.001,0.887,0.002,137.514,0.401,73.416,0.745,235.634,1.031,2002.390,,0.020,Simbad,266.41685,-29.00777,8.747519,1.739881e+31
2,black,S8,16.637,0.182,0.768,0.022,75.057,0.573,337.931,2.120,317.075,0.630,1979.216,,0.037,Simbad,266.41696,-29.00788,13.813865,2.747578e+31
3,black,S9,11.125,0.030,0.791,0.036,81.876,0.458,137.854,0.573,158.079,0.229,1972.924,,0.023,Simbad,266.41690,-29.00790,10.907654,2.169532e+31
4,black,S12,11.962,0.105,0.906,0.003,33.060,0.516,311.173,0.802,236.173,1.146,1995.881,,0.001,Simbad,266.41682,-29.00772,9.633719,1.916147e+31


## modelo de 2 cuerpos (Fuerza central).
Programar la integración numérica (RK4) para 1 sola estrella alrededor de Sgr A* en 3D.
## Generalización a N-Cuerpos
Objetivo: Expandir el modelo para integrar Sgr A* +  5 estrellas interactuando entre sí.
Programar funciones para calcular la Energía Total (
E
=
K
+
U
) y el Momento Angular (
L
→
) en cada paso de tiempo.
Graficar estas cantidades para demostrar empíricamente la conservación (o los límites del error del integrador).
Actualizar el plot de plotly para mostrar múltiples trayectorias (líneas) y las posiciones actuales (marcadores).
Añadir controles de animación o un slider de tiempo si es posible

## Escalado a 49 Estrellas y Análisis de Resultados
Objetivo: Correr la simulación completa y extraer conclusiones.
Inyectar las 49 estrellas al modelo.
Determinar un paso de tiempo (
d
t
) óptimo que no dispare el tiempo de cómputo pero mantenga la precisión en los periapsis (cuando las estrellas pasan muy rápido y cerca del agujero negro).
Escribir en Markdown el análisis: ¿Hay variaciones notables por la interacción entre estrellas comparado con el modelo de 2 cuerpos?
Persona C (Documentación y Refinamiento):
Limpiar el notebook: agrupar variables análogas (x, y, z = pos[0], pos[1], pos[2]), asegurar que todos los bloques de código tengan justificación teórica en Markdown previo.
Añadir anotaciones en Plotly identificando las estrellas más importantes (ej. la estrella S2, si está en su catálogo).


In [ ]:
# ─── Sistema: Sgr A* (en el origen) + 5 estrellas S ───────────────────────
# m en el dict es el parámetro gravitacional G·M (mpc³/yr²)
sistema = [
    dict(m=mu_SgrA, r=[0.0, 0.0, 0.0], v=[0.0, 0.0, 0.0])   # Sgr A*
]

for estado, masa in zip(estados, masas_5):
    sistema.append(dict(
        m = G_canon * masa,
        r = estado[:3].tolist(),
        v = estado[3:].tolist()
    ))

print(f"Sistema con {len(sistema)} cuerpos:")
print(f"  Sgr A*  : μ = {mu_SgrA:.4f} mpc³/yr²")
for s, row in zip(sistema[1:], primeras_5.itertuples()):
    r_norm = np.linalg.norm(s['r'])
    v_norm = np.linalg.norm(s['v'])
    print(f"  {row.Name.strip():5s}: μ = {s['m']:.2e} mpc³/yr², "
          f"|r| = {r_norm:.4f} mpc, |v| = {v_norm:.3f} mpc/yr")

# ─── Integración numérica ──────────────────────────────────────────────────
T_sim   = 100           # años simulados
N_pasos = 5000
ts = np.linspace(0, T_sim, N_pasos)

print(f"\nIntegrando {T_sim} años con {N_pasos} pasos ...")
rs, vs, rps, vps, constantes = pc.ncuerpos_solucion(sistema, ts)
print("Integración completada.")

# ─── Conservación de energía ──────────────────────────────────────────────
# K y U son series temporales; E = K + U debe mantenerse constante
K  = constantes['K']
U  = constantes['U']
E_t = K + U
error_E = np.abs((E_t - E_t[0]) / E_t[0]).max()
print(f"\nError relativo máx. en energía total: {error_E:.2e}")

fig_E, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(ts, (E_t - E_t[0]) / np.abs(E_t[0]))
axs[0].set_xlabel("Tiempo (años)")
axs[0].set_ylabel("(E - E₀) / |E₀|")
axs[0].set_title("Conservación de energía")
axs[0].grid(True)

axs[1].plot(ts, K, label='K', color='tab:orange')
axs[1].plot(ts, U, label='U', color='tab:blue')
axs[1].plot(ts, E_t, label='E = K+U', color='tab:green', linewidth=2)
axs[1].set_xlabel("Tiempo (años)")
axs[1].set_ylabel("Energía (mpc²/yr²)")
axs[1].set_title("Energía cinética, potencial y total")
axs[1].legend()
axs[1].grid(True)
plt.tight_layout()
plt.show()

# ─── Visualización 3D con Plotly ──────────────────────────────────────────
nombres = ['Sgr A*'] + primeras_5['Name'].str.strip().tolist()
fig = pc.plot_ncuerpos_3d(rs, vs, tipo='plotly')

for i, nombre in enumerate(nombres):
    fig.data[i].name = nombre

pc.axis_equal(fig)
fig.update_layout(
    title=f"Sgr A* + 5 estrellas S — simulación de {T_sim} años",
    scene=dict(
        xaxis_title="x (mpc)",
        yaxis_title="y (mpc)",
        zaxis_title="z (mpc)"
    )
)
fig.show()